# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 1: Comparação Geral de Desempenho (Sem Opção de Rejeição)

Neste notebook, é apresentada a análise comparativa global entre os modelos baselines estabelecidos no artigo (*Silva et al., 2025*) e a proposta baseada em **Support Vector Machines (SVM) com Aproximação de Nyström** nas representações espectrais **MEL** e **LOFAR**.


### Parâmetros de Configuração e Treinamento do SVM Nyström:
Para garantir a replicabilidade científica dos experimentos, as seguintes configurações de hiperparâmetros foram empregadas no estimador:
* **Aproximação de Kernel:** Kernel Gaussiano RBF aproximado pelo método de Nyström com $m = 4000$ componentes espectrais.
* **Custo de Regularização ($C$):** $C = 2.0$.
* **Regularização / Penalidade:** ElasticNet com razão $L_1 = 0.15$ (propiciando esparsidade seletiva) e $L_2 = 0.85$.
* **Tratamento de Dimensionalidade (MEL):** PCA desativado (preservação das 256 bandas Mel como entrada direta do estimador).
* **Tratamento de Dimensionalidade (LOFAR):** PCA ativado (projeção linear redutiva prévia para $n_{components} = 64$ para filtragem de ruído oceânico).


### Tabela 1: Métricas de Desempenho Geral no Conjunto de Teste (Sem Rejeição)

| Classificador / Arquitetura | Representação Espectral | Índice SP (%) | Acurácia Global (ACC) (%) | F1-Score (Micro) (%) |
| :--- | :--- | :---: | :---: | :---: |
| **RF** | MEL | 62.22 ± 1.86 | 62.64 ± 1.84 | 63.79 ± 1.73 |
| **RF** | LOFAR | 56.92 ± 1.69 | 58.87 ± 1.68 | 58.00 ± 1.41 |
| **MLP** | MEL | 63.38 ± 1.81 | 64.51 ± 1.75 | 62.89 ± 1.68 |
| **MLP** | LOFAR | 66.51 ± 1.39 | 67.48 ± 1.24 | 66.72 ± 1.17 |
| **CNN** | MEL | 63.52 ± 2.26 | 64.99 ± 2.09 | 63.04 ± 2.02 |
| **CNN** | LOFAR | 66.05 ± 1.90 | 67.02 ± 1.78 | 66.29 ± 2.13 |
| **SVM** *(m=4000, C=2.0, ElasticNet)* | MEL | **63.78 ± 1.14** | **64.56 ± 1.18** | **64.32 ± 1.02** |
| **SVM** *(m=4000, C=2.0, PCA64, ElasticNet)* | LOFAR | **63.10 ± 2.19** | **64.04 ± 1.88** | **64.34 ± 2.23** |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configurações estéticas para gráficos acadêmicos premium
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

### 1. Definição do Conjunto de Dados
Os resultados obtidos sob o protocolo de **Validação Cruzada 5x2 (10 folds)** com a restrição de *Exclusive Ships on Test* são representados programaticamente no pandas DataFrame abaixo.


In [ ]:
data = {
    'Modelo': [
        'RF Mel', 'RF Lofar', 
        'MLP Mel', 'MLP Lofar', 
        'CNN Mel', 'CNN Lofar', 
        'SVM Mel', 'SVM Lofar'
    ],
    'Representacao': ['MEL', 'LOFAR', 'MEL', 'LOFAR', 'MEL', 'LOFAR', 'MEL', 'LOFAR'],
    'SP': [62.22, 56.92, 63.38, 66.51, 63.52, 66.05, 63.78, 63.10],
    'SP_std': [1.86, 1.69, 1.81, 1.39, 2.26, 1.90, 1.14, 2.19],
    'ACC': [62.64, 58.87, 64.51, 67.48, 64.99, 67.02, 64.56, 64.04],
    'ACC_std': [1.84, 1.68, 1.75, 1.24, 2.09, 1.78, 1.18, 1.88]
}

df = pd.DataFrame(data)
df

### 2. Visualização das Métricas Globais (MEL vs LOFAR)
Um gráfico de barras comparando a **Acurácia Global (ACC)** e o **Índice SP (Robustez)** com barras de desvio padrão é plotado a seguir.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

colors_mel = ['#34495e', '#2980b9', '#27ae60', '#e74c3c']
colors_lofar = ['#7f8c8d', '#3498db', '#2ecc71', '#e74c3c']

# Gráfico 1: Acurácia Global
mel_mask = df['Representacao'] == 'MEL'
lofar_mask = df['Representacao'] == 'LOFAR'

x = np.arange(4)
width = 0.35

rects1 = ax1.bar(x - width/2, df[mel_mask]['ACC'], width, yerr=df[mel_mask]['ACC_std'], 
                label='MEL', color='#1abc9c', edgecolor='black', capsize=5, alpha=0.9)
rects2 = ax1.bar(x + width/2, df[lofar_mask]['ACC'], width, yerr=df[lofar_mask]['ACC_std'], 
                label='LOFAR', color='#34495e', edgecolor='black', capsize=5, alpha=0.9)

ax1.set_ylabel('Acurácia Global (%)')
ax1.set_title('Acurácia Global (ACC) por Modelo e Extrator')
ax1.set_xticks(x)
ax1.set_xticklabels(['RF', 'MLP', 'CNN', 'SVM (m=4000)'])
ax1.set_ylim(50, 75)
ax1.legend()

# Gráfico 2: Índice SP
rects3 = ax2.bar(x - width/2, df[mel_mask]['SP'], width, yerr=df[mel_mask]['SP_std'], 
                label='MEL', color='#e67e22', edgecolor='black', capsize=5, alpha=0.9)
rects4 = ax2.bar(x + width/2, df[lofar_mask]['SP'], width, yerr=df[lofar_mask]['SP_std'], 
                label='LOFAR', color='#2c3e50', edgecolor='black', capsize=5, alpha=0.9)

ax2.set_ylabel('Índice SP (%)')
ax2.set_title('Índice SP (Sensibilidade Equilibrada) por Modelo')
ax2.set_xticks(x)
ax2.set_xticklabels(['RF', 'MLP', 'CNN', 'SVM (m=4000)'])
ax2.set_ylim(50, 75)
ax2.legend()

plt.tight_layout()
plt.show()

### 3. Discussão Científica e Conclusões:
1. **Convergência de Capacidade:** O modelo proposto **SVM Nyström (MEL)** alcançou **64.56% ± 1.18% de Acurácia**, assemelhando-se estatisticamente à CNN convolucional profunda (**64.99%**), contudo com **quase metade da variância fold-wise** ($\sigma_{SVM} = 1.18\%$ vs. $\sigma_{CNN} = 2.09\%$). Desse modo, a estabilidade matemática da formulação convexa é evidenciada.
2. **Divergência Crítica do PCA:** Foi verificado que a projeção PCA linear acarreta degradação sobre o extrator MEL (devido à compressão redundante linear sobre eixos logarítmicos pré-integrados). No entanto, o PCA demonstrou-se essencial no LOFAR, atuando como um excelente filtro de ruído caótico tridimensional e propiciando a separabilidade geométrica para o kernel Gaussiano RBF.
